In [1]:
###################################################OWN IMPORT###################################################
from LIDstuff.BAGGING_TESTS.experiment_class import *
from LIDstuff.BAGGING_TESTS.LID_Bagging import *

In [24]:
map  =  {
    'bagging_method:bag | pre_smooth:False | post_smooth:False': 'Simple bagging',
    'bagging_method:bag | pre_smooth:False | post_smooth:True': 'Simple bagging with pre-smoothing',
    'bagging_method:bag | pre_smooth:True | post_smooth:False': 'Simple bagging with post-smoothing',
    'bagging_method:bag | pre_smooth:True | post_smooth:True': 'Simple bagging with pre-smoothing and post-smoothing',
    'bagging_method:None | pre_smooth:False | post_smooth:False': 'Baseline',
    'bagging_method:None | pre_smooth:False | post_smooth:True': 'Baseline with smoothing'}

In [30]:
def unordered_lookup(query, original_map = None, sep= '|'):
    if original_map is None:
        original_map  =  {
    'bagging_method:bag | pre_smooth:False | post_smooth:False': 'Simple bagging',
    'bagging_method:bag | pre_smooth:False | post_smooth:True': 'Simple bagging with pre-smoothing',
    'bagging_method:bag | pre_smooth:True | post_smooth:False': 'Simple bagging with post-smoothing',
    'bagging_method:bag | pre_smooth:True | post_smooth:True': 'Simple bagging with pre-smoothing and post-smoothing',
    'bagging_method:None | pre_smooth:False | post_smooth:False': 'Baseline',
    'bagging_method:None | pre_smooth:False | post_smooth:True': 'Baseline with smoothing'}
    def build_canonical_map(original: dict[str, str], sep: str = '|') -> dict[tuple[str, ...], str]:
        return {
            tuple(sorted(part.strip() for part in key.split(sep))): value
            for key, value in original.items()
        }
    canonical_map = build_canonical_map(original_map)
    signature = tuple(sorted(part.strip() for part in query.split(sep)))
    print(signature)
    return canonical_map.get(signature)

In [31]:
print(unordered_lookup('post_smooth:True | pre_smooth:False | bagging_method:None', map))

('bagging_method:None', 'post_smooth:True', 'pre_smooth:False')
Baseline with smoothing


In [2]:
lit = [0.6, 0.42857142857142855, 0.3, 0.21428571428571427, 0.15789473684210525, 0.11538461538461539, 0.08108108108108109, 0.058823529411764705, 0.041666666666666664]
print([round(l, 3) for l in lit])

[0.6, 0.429, 0.3, 0.214, 0.158, 0.115, 0.081, 0.059, 0.042]


In [2]:
all = ['M1_Sphere', 'M2_Affine_3to5', 'M3_Nonlinear_4to6', 'M4_Nonlinear', 'M5b_Helix2d', 'M6_Nonlinear',
            'M7_Roll', 'M8_Nonlinear', 'M9_Affine', 'M10a_Cubic', 'M10b_Cubic', 'M10c_Cubic', 'M11_Moebius',
            'M12_Norm', 'M13a_Scurve', 'Mn1_Nonlinear', 'Mn2_Nonlinear', 'lollipop_', 'uniform']

param_dicts1 = {'dataset_name': 'M1_Sphere',
                'n': 2500,
                'lid': None,
                'dim': None,
                'estimator_name': 'mle',
                'bagging_method': None,
                'submethod_0': '0',
                'submethod_error': 'log_diff',
                'k': 20,
                'sr': 0.3,
                'Nbag': 10,
                'pre_smooth': False,
                'post_smooth': False,
                't': 2}

In [3]:
exp1 = LID_experiment(params=param_dicts1)

In [4]:
_ = exp1.generate_data(load=True)

In [5]:
_ = exp1.estimate()

In [6]:
sphere_data = exp1.data

In [7]:
exp1.lid_estimates

array([ 8.00599944, 10.75108922,  6.78937315, ..., 10.9510027 ,
       12.30406703, 10.237292  ])

In [8]:
lid_estimates = MLE_LID_Estimator(sphere_data[0], subsample_indexes=None, neighbourhood_size = 20, return_smoothed=False)

In [43]:
lid_estimates

array([ 8.00599944, 10.75108922,  6.78937315, ..., 10.9510027 ,
       12.30406703, 10.237292  ])

In [9]:
np.mean(lid_estimates - exp1.lid_estimates)

5.080380560684716e-16

In [ ]:
param_dicts2 = {'dataset_name': 'M1_Sphere',
                'n': 2500,
                'lid': None,
                'dim': None,
                'estimator_name': 'mle',
                'bagging_method': 'bag',
                'submethod_0': '0',
                'submethod_error': 'log_diff',
                'k': 10,
                'sr': 0.3,
                'Nbag': 10,
                'pre_smooth': False,
                'post_smooth': False,
                't': 2}

In [27]:
exp2 = LID_experiment(params=param_dicts2)
_ = exp2.generate_data(load=True)
_ = exp2.estimate()

In [28]:
sphere_data = exp1.data[0]

In [29]:
lid_estimates2 = lid_Bagging(sphere_data, ensemble_size = 10, subsample_rate = 0.3, rand_gen_seed = 1, return_OOB_LE = False,
                             LE_type = "SNL", NN_size_for_w = 10, dist_assumption = "Tail", lid_Estimator = MLE_LID_Estimator, neighbourhood_size = 10, return_smoothed=False)

In [32]:
np.mean(lid_estimates2[0]-exp2.lid_estimates)**2

0.003792925459504065

In [46]:
k = 10
sr = 0.3
Nbag = 10

param_dicts2 = {'dataset_name': 'M1_Sphere',
                'n': 2500,
                'lid': None,
                'dim': None,
                'estimator_name': 'mle',
                'bagging_method': 'bag',
                'submethod_0': '0',
                'submethod_error': 'log_diff',
                'k': k,
                'sr': sr,
                'Nbag': Nbag,
                'pre_smooth': False,
                'post_smooth': False,
                't': 2}

In [48]:
from tqdm.notebook import tqdm

testlist = []
for i in tqdm(range(5)):
    exp2 = LID_experiment(params=param_dicts2)
    _ = exp2.generate_data(load=True)
    _ = exp2.estimate()
    sphere_data = exp2.data[0]
    lid_estimates2 = lid_Bagging(sphere_data, ensemble_size = Nbag, subsample_rate = sr, rand_gen_seed = i, return_OOB_LE = False,
                                 LE_type = "SNL", NN_size_for_w = k, dist_assumption = "Tail", lid_Estimator = MLE_LID_Estimator, neighbourhood_size = k, return_smoothed=False)
    testlist.append(np.abs(np.mean(lid_estimates2[0]-exp2.lid_estimates)))
print(np.mean(testlist))

testlist = []
for i in tqdm(range(5)):
    exp2 = LID_experiment(params=param_dicts2)
    _ = exp2.generate_data(load=True)
    _ = exp2.estimate()
    
    exp3 = LID_experiment(params=param_dicts2)
    _ = exp3.generate_data(load=True)
    _ = exp3.estimate()

    testlist.append(np.abs(np.mean(exp3.lid_estimates-exp2.lid_estimates)))
print(np.mean(testlist))

testlist = []
for i in tqdm(range(5)):
    exp2 = LID_experiment(params=param_dicts2)
    _ = exp2.generate_data(load=True)
    sphere_data = exp2.data[0]
    lid_estimates2 = lid_Bagging(sphere_data, ensemble_size = Nbag, subsample_rate = sr, rand_gen_seed = i, return_OOB_LE = False,
                                 LE_type = "SNL", NN_size_for_w = k, dist_assumption = "Tail", lid_Estimator = MLE_LID_Estimator, neighbourhood_size = k, return_smoothed=False)
    lid_estimates3 = lid_Bagging(sphere_data, ensemble_size = Nbag, subsample_rate = sr, rand_gen_seed = i+1, return_OOB_LE = False,
                             LE_type = "SNL", NN_size_for_w = k, dist_assumption = "Tail", lid_Estimator = MLE_LID_Estimator, neighbourhood_size = k, return_smoothed=False)
    testlist.append(np.abs(np.mean(lid_estimates2[0]-lid_estimates3[0])))
print(np.mean(testlist))

  0%|          | 0/5 [00:00<?, ?it/s]

0.02247812614223311


  0%|          | 0/5 [00:00<?, ?it/s]

0.028985762514142005


  0%|          | 0/5 [00:00<?, ?it/s]

0.015092958336835726


In [51]:
from LIDstuff.BAGGING_TESTS.RunningEstimators.BaseEstimators import *
exp2 = LID_experiment(params=param_dicts2)
_ = exp2.generate_data(load=True)
sphere_data = exp2.data[0]
for i in tqdm(range(5)):
    two_NN_LID_Estimator(sphere_data, subsample_indexes = None, neighbourhood_size = 100, perc_deleted = 0, return_smoothed = False)
for i in tqdm(range(5)):
    sk_2NN_full(sphere_data, k = 100, correct = False, dists=None, knnidx=None, w=None, smooth=False, geo=None)

100%|██████████| 5/5 [00:25<00:00,  5.13s/it]


In [55]:
for i in tqdm(range(5)):
    MLE_LID_Estimator(sphere_data, subsample_indexes = None, neighbourhood_size = 10, return_smoothed=False)
for i in tqdm(range(5)):
    sk_MLE_full(sphere_data, k = 10, correct = False, dists=None, knnidx=None, w=None, smooth=False, geo=None)

100%|██████████| 5/5 [00:01<00:00,  3.84it/s]


In [53]:
sk_2NN(X, dists, knnidx, k = 10, correct = True, w = None, return_ks = False, use_w = 'indirect', smooth=False, geo=None)

NameError: name 'X' is not defined